# 증강

In [2]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
from tqdm import tqdm
import re
import os
import json
from glob import glob
import pandas as pd

### 1. nllb -> 실패 (번역하면서 요약도 해버림)

In [ ]:
# ---------------------------
# 1. 모델 로드 (NLLB-200)
# ---------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"

model_name = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

In [ ]:
# ---------------------------
# 2. 스페셜 토큰 보호용 함수
# ---------------------------
# 스페셜 토큰 목록
SPECIAL_TOKENS = [
    '#Person1#', '#Person2#', '#Person3#', '#Person4#', '#Person5#', '#Person6#', '#Person7#',
    '#PhoneNumber#', '#Address#', '#PassportNumber#'
]

def protect_special_tokens(text, special_tokens=SPECIAL_TOKENS):
    """
    모든 스페셜 토큰을 __ST{index}__ 형태로 보호한다.
    예: #Person1# → __ST0__,  #PhoneNumber# → __ST7__
    """
    protected_text = text
    for idx, token in enumerate(special_tokens):
        placeholder = f"__ST{idx}__"
        protected_text = protected_text.replace(token, placeholder)
    return protected_text


def restore_special_tokens(text, special_tokens=SPECIAL_TOKENS):
    """
    __ST{index}__ 형태의 placeholder를 다시 원래 스페셜 토큰으로 복원한다.
    """
    restored_text = text
    for idx, token in enumerate(special_tokens):
        placeholder = f"__ST{idx}__"
        restored_text = restored_text.replace(placeholder, token)
    return restored_text


In [ ]:
# ---------------------------
# 3. NLLB 번역 함수
# ---------------------------
def translate(text, src_lang, tgt_lang, max_length=512):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_length
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            forced_bos_token_id=tokenizer.lang_code_to_id[tgt_lang],
            max_length=max_length
        )

    return tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]


def back_translate(text):

    print("text : " + text)
    
    """
    한국어 → 영어 → 한국어
    """
    # Step 1: ko -> en
    en = translate(text, "kor_Hang", "eng_Latn")

    print("en : " + en)

    # Step 2: en -> ko
    ko = translate(en, "eng_Latn", "kor_Hang")

    print("ko : " + ko)
    return ko


In [ ]:
# ---------------------------
# 4. train.csv 불러오기
# ---------------------------
data_dir = "../data/processed/"
train_data_path = os.path.join(data_dir, "train_preprocessed.csv")
train_data = pd.read_csv(train_data_path)
print(f"✅ 학습 데이터 개수: {len(train_data)}")

In [ ]:
# ---------------------------
# 5. dialogue만 back-translation
# ---------------------------
aug_dialogues = []

print("\n🔄 Back-translation 진행 중...")

for dialogue in tqdm(train_data["dialogue"]):

    # 1) 스페셜 토큰 보호
    protected = protect_special_tokens(dialogue)

    # 2) back-translation
    bt = back_translate(protected)

    # 3) 원래 스페셜 토큰 복구
    restored = restore_special_tokens(bt)

    aug_dialogues.append(restored)

### 2. GoogleTranslator -> 실패 (번역 품질 구림)

In [ ]:
!pip install deep-translator

In [ ]:
from deep_translator import GoogleTranslator

def back_translation(text, src_lang='ko', intermediate_lang='en'):
    try:
        translated = GoogleTranslator(source=src_lang, target=intermediate_lang).translate(text)
        
        back_translated = GoogleTranslator(source=intermediate_lang, target=src_lang).translate(translated)
        
        return back_translated
    except Exception as e:
        return text  # 오류 발생 시 원본 텍스트 반환

In [ ]:
aug_dialogues2 = []

# train_data의 1행만 로드
train_data2 = pd.read_csv("../data/processed/train_preprocessed.csv").head(1)

for dialogue in tqdm(train_data2["dialogue"]):

    # 1) 스페셜 토큰 보호
    protected = protect_special_tokens(dialogue)

    # 2) back-translation
    bt = back_translation(protected)

    # 3) 원래 스페셜 토큰 복구
    restored = restore_special_tokens(bt)

    aug_dialogues2.append(restored)

### 3. Solar API

In [ ]:
import requests
from openai import OpenAI # openai==1.52.2
import time

SOLAR_API_KEY = 'up_XLatisRvN9uTXyffIoXNfNGxQAnKE'
SOLAR_URL = "https://api.upstage.ai/v1"


def solar_translate(text, src_lang, tgt_lang):

    client = OpenAI(
        api_key=SOLAR_API_KEY,
        base_url=SOLAR_URL
    )
    
    prompt = (
        f"Translate the following text from {src_lang} to {tgt_lang}. "
        f"Do not change formatting, punctuation, or structure.\n\n"
        f"Text:\n{text}"
    )

    try:
        response = client.chat.completions.create(
            model="solar-mini",   # 빠르고 저렴하며 구조 유지 잘함
            messages=[
                {"role": "system", "content": "You are a precise translation model. Follow rules exactly."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.0   # 번역/증강 일관성 유지 위해 반드시 0
        )

        return response.choices[0].message.content.strip()

    except Exception as e:
        print(f"Error: {e}")
        time.sleep(1)
        return None
        
    

In [64]:
def solar_back_translate_ko_en_ko(text):

    protected = protect_special_tokens(text)
    
    """
    한국어 → 영어 → 한국어 back-translation
    """
    # Step 1: Korean → English
    en = solar_translate(protected, src_lang="Korean", tgt_lang="English")
    # Step 2: English → Korean
    back_translated = solar_translate(en, src_lang="English", tgt_lang="Korean")

    restored = restore_special_tokens(back_translated)

    return restored, en

In [ ]:
train_data = pd.read_csv("../data/processed/train_preprocessed.csv").head(3)

aug_dialogues = []

for i in range(len(train_data)):
    back_translated, english_text = solar_back_translate_ko_en_ko(train_data["dialogue"].iloc[i])
    aug_dialogues.append(back_translated)



In [68]:
# ---------------------------
# 6. 새로운 증강 데이터셋 생성
# ---------------------------
df_aug = pd.DataFrame({
    "fname": train_data["fname"] + "_bt",
    "dialogue": aug_dialogues,
    "summary": train_data["summary"]  # summary는 원본 그대로 유지
})

In [69]:
# ---------------------------
# 7. 증강본 저장
# ---------------------------
data_dir = "../data/augmented/"
now = time.strftime("%Y%m%d_%H%M%S")
df_aug.to_csv(os.path.join(data_dir, f"train_aug_{now}.csv"), index=False)

print(f"\n✅ 완료! train_aug_{now}.csv 생성됨.")


✅ 완료! train_aug_20251206_021513.csv 생성됨.


In [10]:
# 원본 train 데이터의 dialogue에서 영어로 된 단어를 찾아 fname, 단어 찾아오기
train_orig = pd.read_csv("../data/processed/train_preprocessed.csv")
SPECIAL_TOKENS = [
    '#Person1#', '#Person2#', '#Person3#', '#Person4#', '#Person5#', '#Person6#', '#Person7#',
    '#PhoneNumber#', '#Address#', '#PassportNumber#'
]
eng_words = []
for idx, row in train_orig.iterrows():
    original_dialogue = row['dialogue']      # fname 찾을 때 사용할 원본
    cleaned_dialogue = original_dialogue     # 영어 단어 탐색용 복사본

    # 스페셜 토큰 제거
    for token in SPECIAL_TOKENS:
        cleaned_dialogue = cleaned_dialogue.replace(token, "")
    
    # 영어 단어 추출
    words = re.findall(r'[a-zA-Z]+', cleaned_dialogue)

    # 영어 단어가 존재하면 (fname, words) 추가
    if words:
        eng_words.append((row['fname'], words))

print(eng_words)

[('train_0', ['Mr', 'Smith', 'Dr', 'Hawkins', 'Mr', 'Smith']), ('train_1', ['Mrs', 'Parker', 'Dr', 'Peters', 'Ricky', 'Ricky', 'B', 'A', 'Ricky']), ('train_4', ['Malik', 'Wen', 'Nikki']), ('train_5', ['Aims', 'Lisa']), ('train_7', ['Judy', 'Liao', 'L', 'I', 'A', 'O', 'Judy', 'Liao', 'Judy', 'Liao']), ('train_8', ['CPU', 'MB', 'DVD']), ('train_11', ['Miami', 'University']), ('train_12', ['Bean']), ('train_13', ['Keith', 'James', 'Keith']), ('train_14', ['CD']), ('train_17', ['John', 'John']), ('train_24', ['Superbad', 'Superbad', 'Superbad', 'DVD']), ('train_27', ['Mr', 'White', 'Jessica', 'Mr', 'White', 'Jessica']), ('train_33', ['Reader', 's', 'Guide', 'to', 'Periodical', 'Literature']), ('train_37', ['Sue', 'Snowdon']), ('train_40', ['TV']), ('train_42', ['Smith', 'Smith', 'Smith', 'Jim', 'White', 'White', 'Smith']), ('train_45', ['Nancy', 'TV']), ('train_46', ['East', 'York']), ('train_47', ['Ahmed', 'Carla']), ('train_48', ['Grove', 'Street']), ('train_50', ['James']), ('train_52',

In [3]:
# 원본 + 증강본 합치기
def load_all_checkpoints(ckpt_dir="../data/augmented/"):
    df_orig = pd.read_csv("../data/processed/train_preprocessed.csv")
    df_aug = pd.read_csv(os.path.join(ckpt_dir, "train_aug_solar_mini_full_cleaned.csv"))
    
    df_merged = pd.concat([df_orig, df_aug], ignore_index=True)
    df_merged = df_merged.sample(frac=1, random_state=42).reset_index(drop=True)
    df_merged.to_csv(os.path.join(ckpt_dir, "train_augmented_full.csv"), index=False, encoding="utf-8-sig")

    print(f"\n✅ 총 증강 데이터 개수: {len(df_merged)}")
    return df_merged

In [4]:
load_all_checkpoints()


✅ 총 증강 데이터 개수: 24902


,fname,dialogue,summary,topic
0,train_9423,"#Person1#: 날씨 진짜 좋다.\n#Person2#: 응, 맞아.\n#Pers...","#Person2#는 비가 오면 공기가 맑아지는 것을 좋아하고, 비 온 후에 별을 더...",비 오는 날의 장점
1,train_4534,"#Person1#: 스페인 비행기표는 예약했어?\n#Person2#: 음, 이번에는...",#Person1#와 #Person2#는 스페인 여행을 계획하면서 매 순간을 즐기기 ...,배로 떠나는 스페인 여행
2,train_5598,"#Person1#: 봄 축제가 다가오는데, 어떻게 축하하고 싶어, Danny?\n#...",#Person1#은 자신의 나라에서 봄 축제 때 아이들이 새 옷을 입고 어르신들에게...,봄 축제와 가족모임
3,train_11059_bt,"#Person1#: 안녕, Betty! \n#Person2#: 안녕, Andy. 새...",Andy는 Betty에게 새 회사의 기업 문화가 이전 직장과 완전히 다르다고 말합니...,NaN
4,train_8890_bt,"#Person1#: 자, 스미스씨, 이렇게 또 오셨군요. 이번 달에는 네번째 방문이...",Smith 씨는 약을 복용하고 운동을 시도하고 있지만 상태가 좋지 않다고 말합니다....,NaN
...,...,...,...,...
24897,train_9128_bt,"#Person1#: 안녕하세요, 존슨씨. 더 필요한 약은 없으세요?\n#Person...","Johnson 씨는 침대가 불편하다고 느꼈고, #Person1#은 침대의 여러 부분...",NaN
24898,train_5390,#Person1#: 가게에 좀 다녀와 줄 수 있어?\n#Person2#: 물론이지....,#Person1#은 #Person2#에게 필요한 물건을 사오고 처방전을 받아달라고 ...,장보기 및 처방전 수령
24899,train_860,#Person1#: 내 사무실에 이 잡지 더미 누가 둔 거야?\n#Person2#:...,#Person2#는 #Person1#에게 Alice가 잡지들을 도서관에 가져다달라고...,일상적인 대화
24900,train_3342_bt,"#Person1#: 선생님, 아직 등록 안 하셨나요?\n#Person2#: 네, 아...",#Person2#가 병원에 입원 신청을 했으나 빈 병상이 없어서 기다려야 합니다. ...,NaN


In [1]:
print(f"\n✅ 총 증강 데이터 개수: {len(df_merged)}")

NameError: name 'df_merged' is not defined